# 01 — Main Pipeline: HRNet 2D → 4D-Humans 3D

**Run on Colab** (T4 / V100 / A100).
Mac cannot install detectron2 cleanly; this notebook will fail outside Linux+CUDA.

Pipeline:
```
Input → ViTDet → HRNet-W48 (2D) → 4D-Humans HMR2 (3D SMPL mesh) → quad-plot + GIF
```

Expected runtime: ~7s for 5 images on Colab T4 (per v3 plan §C.1).

## Cell 1 — Environment (one-time, ~3 min)

In [ ]:
# Headless rendering deps
!apt-get install -q -y libosmesa6-dev freeglut3-dev libglfw3-dev
import os; os.environ['PYOPENGL_PLATFORM'] = 'osmesa'

# Pinned torch + MMPose stack
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -U openmim
!mim install -q "mmengine>=0.7" "mmcv>=2.0,<2.2" "mmdet>=3.1" "mmpose>=1.1"

# 4D-Humans (HMR 2.0)
!git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -q -e '.[all]'
%cd /content

# SMPL family + viz
!pip install -q smplx chumpy trimesh pyrender imageio matplotlib pandas

# Mount Drive (where SMPL .pkl + SMPLX .npz live)
from google.colab import drive
drive.mount('/content/drive')

# 4D-Humans expects this exact filename in its data/ dir
!mkdir -p /content/4D-Humans/data
!cp /content/drive/MyDrive/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl /content/4D-Humans/data/

# Auto-download HMR 2.0 weights
from hmr2.utils.download_util import download_models, CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
print("Setup complete ✓")

## Cell 2 — HRNet 2D inference

In [ ]:
import time
from pathlib import Path
import numpy as np
from mmpose.apis import MMPoseInferencer

TEST_IMAGES = sorted(Path('/content/test_images').glob('*.jpg'))
print(f"{len(TEST_IMAGES)} test images")

inferencer = MMPoseInferencer(
    pose2d='td-hm_hrnet-w48_8xb32-210e_coco-256x192',
    det_model='rtmdet-m', device='cuda')

results_2d = {}
for img_path in TEST_IMAGES:
    t0 = time.time()
    res = next(inferencer(str(img_path), return_vis=True, vis_out_dir='/content/results/2d_vis/'))
    rt = time.time() - t0
    preds = res['predictions'][0]
    results_2d[img_path.name] = {
        'kpts': np.array([p['keypoints'] for p in preds]) if preds else np.zeros((0,17,2)),
        'scores': np.array([p['keypoint_scores'] for p in preds]) if preds else np.zeros((0,17)),
        'runtime_s': rt,
    }
    print(f"  {img_path.name}: {len(preds)} persons, {rt*1000:.0f} ms")

# Free HRNet memory before loading 4D-Humans
del inferencer
import torch; torch.cuda.empty_cache()

## Cell 3 — 4D-Humans (HMR 2.0) inference

In [ ]:
# This cell mirrors 4D-Humans/demo.py — copy the canonical detector setup
# from there if your repo version differs.
import torch, cv2
from hmr2.models import load_hmr2, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils.utils_detectron2 import DefaultPredictor_Lazy
from detectron2.config import LazyConfig

model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.cuda().eval()

# Load ViTDet detector exactly as 4D-Humans/demo.py does
det_cfg = LazyConfig.load(str(Path('/content/4D-Humans/hmr2/config') /
                              'cascade_mask_rcnn_vitdet_h_75ep.py'))
det_cfg.train.init_checkpoint = "https://dl.fbaipublicfiles.com/detectron2/ViTDet/COCO/cascade_mask_rcnn_vitdet_h/f328730692/model_final_f05665.pkl"
for i in range(3):
    det_cfg.model.roi_heads.box_predictors[i].test_score_thresh = 0.5
detector = DefaultPredictor_Lazy(det_cfg)

results_3d = {}
for img_path in TEST_IMAGES:
    img = cv2.imread(str(img_path))
    inst = detector(img)['instances']
    boxes = inst.pred_boxes.tensor.cpu().numpy()
    boxes = boxes[inst.pred_classes.cpu().numpy() == 0]
    if len(boxes) == 0:
        results_3d[img_path.name] = {'verts': np.zeros((0,6890,3)), 'cam_t': np.zeros((0,3)),
                                       'boxes': np.zeros((0,4)), 'runtime_s': 0.0}
        continue

    ds = ViTDetDataset(model_cfg, img, boxes)
    dl = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)

    t0 = time.time()
    vs, ts = [], []
    for batch in dl:
        batch = recursive_to(batch, 'cuda')
        with torch.no_grad():
            o = model(batch)
        vs.append(o['pred_vertices'].cpu().numpy())
        ts.append(o['pred_cam_t'].cpu().numpy())
    results_3d[img_path.name] = {
        'verts': np.concatenate(vs, 0),
        'cam_t': np.concatenate(ts, 0),
        'boxes': boxes,
        'runtime_s': time.time() - t0,
    }
    print(f"  {img_path.name}: {len(boxes)} persons, {results_3d[img_path.name]['runtime_s']*1000:.0f} ms")

## Cell 4 — Quad-plot + rotation GIF (per image)

In [ ]:
import sys; sys.path.insert(0, '/content')
from demo.src.visualize import quad_plot, rotation_gif, pick_center_person
import cv2

for img_path in TEST_IMAGES:
    img = cv2.imread(str(img_path))[:, :, ::-1]
    vis_path = next(Path('/content/results/2d_vis').glob(f'{img_path.stem}*'), None)
    if vis_path is None: continue
    overlay = cv2.imread(str(vis_path))[:, :, ::-1]

    verts_all = results_3d[img_path.name]['verts']
    cam_t_all = results_3d[img_path.name]['cam_t']
    idx = pick_center_person(verts_all, cam_t_all, img.shape)
    if idx is None: continue
    verts = verts_all[idx]

    # SMPL has 6890 vertices, 13776 faces; load faces from 4D-Humans
    smpl_faces = model.smpl.faces

    quad_plot(img, overlay, verts, smpl_faces, f'/content/results/quadplot_{img_path.stem}.png')
    rotation_gif(verts, smpl_faces, f'/content/results/rotation_{img_path.stem}.gif')

print('All quad-plots + GIFs ready in /content/results/')

## Cell 5 — Runtime table (the headline result)

In [ ]:
from demo.src.compare import runtime_table, save_tables
df = runtime_table(results_2d, results_3d)
print(df.to_string())
save_tables(df, df_agreement=None, out_dir='/content/results')

## ✅ Outputs

In `/content/results/`:
- `quadplot_*.png` — 5 × (input | 2D | 3D front | 3D side)
- `rotation_*.gif` — 5 × 360° mesh rotation
- `runtime_table.csv / .tex` — headline runtime numbers

Download these for the presentation.